# Neural Networks

This homework will be an extension of the lab we did working with CIFAR data (the homework doesn't come bundled with the data since you already have it from lab. If you don't have it you can get it from the lab from this week). We'll convert our network from a binary to a multiclass setting, much like you did with the MNIST data, and we'll experiment with some ways to improve network performance.

NOTE: We'll be talking about PyTorch during the week before this homework is due, but _don't_ use PyTorch for this homework. We'll switch over the PyTorch for the next one, but I think it's important to understand how the machine learning algorithms work under the hood.

First we'll load and normalize the data, just like we had in the lab.

In [1]:
import pickle
import numpy as np

def load_data(dirname):
    data = []
    labels = []
    for i in range(1, 6):
        with open(dirname + '/data_batch_' + str(i), 'rb') as fo:
            obj = pickle.load(fo, encoding='bytes')
            data.append(obj[b'data'])
            labels.append(obj[b'labels'])
    with open(dirname + '/batches.meta', 'rb') as fo:
        names = pickle.load(fo, encoding='bytes')[b'label_names']
    data = np.concatenate(data, axis=0)
    labels = np.concatenate(labels)
    return data.reshape(-1, 3, 32, 32), labels, names

In [2]:
# You may need to change the path below--it should point to the directory where
# the CIFAR data files are held on your system.
images, labels, names = load_data('./networks_lab/data/cifar')
permute = np.random.permutation(labels.shape[0])
images = images[permute]
labels = labels[permute]

# Split it
train_size = int(images.shape[0] * 0.8)
train_images = images[:train_size].reshape(-1, 3*32 * 32)
train_labels = labels[:train_size]
valid_images = images[train_size:].reshape(-1, 3*32 * 32)
valid_labels = labels[train_size:]

mean = np.mean(train_images)
std = np.std(train_images)

norm_train_images = (train_images - mean) / std
norm_valid_images = (valid_images - mean) / std

Now we'll set up and train a neural network to classify these images. There are different options you can choose from at different levels of complexity. The baseline assignment is to define a neural network with at least one hidden layer using ReLU as the activation function. Train your network using the multiclass cross-entropy loss we defined in the previous homework assignment. Completing this part of the assignment (correctly) guarantees at least a C.

Beyond this, I have two lists of possible extensions:

- Minor
  + Add $L_2$ regularization. That is, if your parameters are $\beta$, minimize $L(\beta) + \lambda \| \beta \|_2^2$. Remember that $\| \beta \|_2^2 = \sum_i \beta_i^2$. You'll need to compute the gradient of $\| \beta \|_2^2$ and add it to your gradients for each parameter. You'll need to choose a regularization hyperparameter $\lambda$. I recommend starting around 1e-4. How does this change your results (if at all)? (Note that the regularization in the lab solution is $L_1$ regularization, so this is slightly different.)
  + Use a different activation function to replace the ReLU. This will require computing the derivative of your new activation function and replacing the derivative of ReLU with it. (The derivative of ReLU is the line in the lab `h_grad[h <= 0] = 0`. We've already computed the derivative of $\tanh$ in a previous homework, so that might be a good choice.) How does this change your results (if at all)?
  + Split your data into a training set and a validation set. Track and plot the training accuracy and validation accuracy during training. What (if any) information can you get from this plot?
- Major
  + Define your training function to accept a _list_ of hidden layer sizes and set up the network accordingly. For example, I should be able to pass your function the list `[256, 128]` and it will create a network with two hidden layers of sizes 256 and 128. But if I pass it `[256, 128, 64]` then it should create a network with _three_ hidden layers of appropriate size.
  + Use at least two hidden layers and add data augmentation. That is, for each iteration of training, you should randomly perturb each image independently. A couple of good options are listed below. Implement at least two.
    * Horizontal flip: randomly mirror an image with probability 0.5
    * Grayscale: replace each color value with the average of all three color values in an image with probability 0.5
    * Brightness: choose a factor uniformly at random in some range around one (say 0.9-1.1) then multiply each pixel value by that factor. You can also choose different factors for each color channel.
    * Shift: randomly choose a number of pixels (our images are small so probably around 3 or 4 at most) and a direction. Shift the image by the chosen number of pixels in the chosen direction and fill in the leftover gap with the mean values of each color.
    
Using at least two hidden layers and doing any two of the minor extensions _or_ a major extension (correctly) guarantees a B. Doing all of the minor extensions and at least one of the major extensions (correctly) is good for an A.

In [3]:
# def tanh(x):
#     return np.tanh(x)

# def derivative_tanh(x):
#     return (1 - np.power(x,2))

def relu(x):
    return np.maximum(x, 0)

def softmax(x):
    return np.exp(x)/np.sum(np.exp(x), axis=1, keepdims=True)

def train(xs, ys, valid_xs, valid_ys, epochs=10, lr=5e-4, batch_size=64):
    
    size1 = 512
    size2 = 256
    
    rng = np.random.default_rng()
    w1 = rng.normal(scale=np.sqrt(4/(xs.shape[1] + size1)), size=(xs.shape[1], size1))
    b1 = np.zeros(size1)
    w2 = rng.normal(scale=np.sqrt(4/(size1 + size2)), size=(size1, size2))
    b2 = np.zeros(size2)
    w3 = np.zeros((size2, 10)) # 10 because 10 classes in the end so there will be 10 neuron connections in the output layer
    b3 = np.zeros(10)

    one_hot = np.zeros((ys.shape[0], 10))
    one_hot[np.arange(ys.shape[0]), ys] = 1
    
    num_batches = xs.shape[0] // batch_size
    if xs.shape[0] % batch_size != 0:
        num_batches += 1
        
    for i in range(epochs):
        # Shuffle the data at the start of each epoch
        permute = rng.permutation(xs.shape[0])
        xs = xs[permute]
        ys = ys[permute]
        one_hot = one_hot[permute]
        
        for j in range(num_batches):
            end_index = min(batch_size * (j + 1), xs.shape[0])
            batch_xs = xs[batch_size * j : end_index]
            batch_ys = ys[batch_size * j : end_index]
            batch_one_hot = one_hot[batch_size * j : end_index]
            
            # forward pass computing predictions
            h1 = relu(batch_xs @ w1 + b1)
            h2 = relu(h1 @ w2 + b2)
            logits = (h2 @ w3 + b3)
            
            # softmax for probabilities bc multiclass 
            preds = softmax(logits)
            # this is the multiclass loss from last homework 
            correct_pred = preds[np.arange(preds.shape[0]), batch_ys]
            loss = np.mean(-np.log(correct_pred))
            
            # gradient computations here
            logit_grad = preds - batch_one_hot
            b3_grad = np.sum(logit_grad, axis=0)
            w3_grad = h2.T @ logit_grad
            h2_grad = logit_grad @ w3.T
            h2_grad[h2 <= 0] = 0
            b2_grad = np.sum(h2_grad, axis=0)
            w2_grad = h1.T @ h2_grad
            h1_grad = h2_grad @ w2.T
            h1_grad[h1 <= 0] = 0
            b1_grad = np.sum(h1_grad, axis=0)
            w1_grad = batch_xs.T @ h1_grad
            
            # update model parameters
            w1 -= lr * w1_grad
            b1 -= lr * b1_grad
            w2 -= lr * w2_grad
            b2 -= lr * b2_grad
            w3 -= lr * w3_grad
            b3 -= lr * b3_grad

        h1 = relu(valid_xs @ w1 + b1)
        h2 = relu(h1 @ w2 + b2)
        logits = (h2 @ w3 + b3)
        valid_preds = softmax(logits)
        valid_max = np.argmax(valid_preds, axis=1)
        accuracy = np.mean(valid_max == valid_ys)
        
        print("Epoch:", i, "Loss:", loss, "Validation accuracy:", accuracy)
        
    return w1, b1, w2, b2, w3, b3

# helper functions for the custom hidden layers challenge
def make_weights(xs, hl):
    rng = np.random.default_rng()
    weights = []
    prev = xs.shape[1]
    for i in range(len(hl)):
        w = rng.normal(scale=np.sqrt(4/(prev + hl[i])), size=(prev, hl[i]))
        weights.append(w)
        prev = hl[i]
    w = np.zeros((hl[-1], 10))
    weights.append(w)
    return weights

def make_betas(hl):
    betas = []
    for i in range(len(hl)):
        b = np.zeros(hl[i])
        betas.append(b)
    b = np.zeros(10)
    betas.append(b)
    return betas

# custom hidden layers! has all the same parameters except hl_array which takes the form described in the extension instructions
def train_custom_hl(xs, ys, valid_xs, valid_ys, hl_array, epochs=10, lr=5e-4, batch_size=64):

    num_hls = len(hl_array)
    
    rng = np.random.default_rng()

    weights = make_weights(xs, hl_array)
    betas = make_betas(hl_array)
    
    one_hot = np.zeros((ys.shape[0], 10))
    one_hot[np.arange(ys.shape[0]), ys] = 1
    
    num_batches = xs.shape[0] // batch_size
    if xs.shape[0] % batch_size != 0:
        num_batches += 1
        
    for i in range(epochs):
        # Shuffle the data at the start of each epoch
        permute = rng.permutation(xs.shape[0])
        xs = xs[permute]
        ys = ys[permute]
        one_hot = one_hot[permute]
        
        for j in range(num_batches):
            end_index = min(batch_size * (j + 1), xs.shape[0])
            batch_xs = xs[batch_size * j : end_index]
            batch_ys = ys[batch_size * j : end_index]
            batch_one_hot = one_hot[batch_size * j : end_index]
            
            # forward pass computing predictions
            hl_arr = []
            prev = batch_xs
            for k in range(num_hls):
                hl = relu(prev @ weights[k] + betas[k])
                hl_arr.append(hl)
                prev = hl
            logits = (hl_arr[-1] @ weights[-1] + betas[-1]) 
            
            # softmax & loss (same as above)
            preds = softmax(logits)
            correct_pred = preds[np.arange(preds.shape[0]), batch_ys]
            loss = np.mean(-np.log(correct_pred))
            
            # gradient computations here
            beta_grads = []
            weight_grads = []
            hl_grads = []
            logit_grad = preds - batch_one_hot
            prev = logit_grad
            for l in range(num_hls):
                b_grad = np.sum(prev, axis=0)
                w_grad = hl_arr[len(hl_arr)-l - 1].T @ prev
                hl_grad = prev @ weights[len(weights)-l - 1].T
                hl_grad[hl_arr[len(hl_arr) - l - 1] <= 0] = 0
                beta_grads.append(b_grad)
                weight_grads.append(w_grad)
                hl_grads.append(hl_grad)
                prev = hl_grad
            b_grad = np.sum(hl_grads[-1], axis=0)
            w_grad = batch_xs.T @ hl_grads[-1]
            beta_grads.append(b_grad)
            weight_grads.append(w_grad)
            
            # update model parameters
            for n in range(num_hls + 1):  # +1 for output layer
                weights[n] -= lr * weight_grads[num_hls - n]
                betas[n] -= lr * beta_grads[num_hls - n]
            
        hl_arr_valid = []
        prev = valid_xs
        for m in range(num_hls):
            hl = relu(prev @ weights[m] + betas[m])
            hl_arr_valid.append(hl)
            prev = hl
        logits = (hl_arr_valid[-1] @ weights[-1] + betas[-1])
        valid_preds = softmax(logits)
        valid_max = np.argmax(valid_preds, axis=1)
        accuracy = np.mean(valid_max == valid_ys)
        
        print("Epoch:", i, "Loss:", loss, "Validation accuracy:", accuracy)
        
    return w1, b1, w2, b2, w3, b3

# THIS CALLS THE NORMAL TRAIN FUNCTION
w1, b1, w2, b2, w3, b3 = train(norm_train_images, # the data is gonna go in as an N x 3*32*32 matrix where N is the number of images
                               train_labels,
                               norm_valid_images,
                               valid_labels,
                               epochs=10,
                               lr=6e-4) 

# THIS CALLS THE CUSTOM HIDDEN LAYERS TRAINING FUNCTION
w1, b1, w2, b2, w3, b3 = train_custom_hl(norm_train_images, 
                               train_labels,
                               norm_valid_images,
                               valid_labels,
                                [512, 256, 128, 64], # where this is the hidden layers input
                               epochs=10,
                               lr=3e-4) # jumps around a little but generally goes in the right direction


Epoch: 0 Loss: 1.5057060244775666 Validation accuracy: 0.4452
Epoch: 1 Loss: 1.4318873355225055 Validation accuracy: 0.4773
Epoch: 2 Loss: 1.4192820345622201 Validation accuracy: 0.4799
Epoch: 3 Loss: 1.1825227288157105 Validation accuracy: 0.4961
Epoch: 4 Loss: 0.9714678570323109 Validation accuracy: 0.4978
Epoch: 5 Loss: 0.9895565403687152 Validation accuracy: 0.4944
Epoch: 6 Loss: 0.855161109054166 Validation accuracy: 0.5034
Epoch: 7 Loss: 0.9433038603223964 Validation accuracy: 0.5098
Epoch: 8 Loss: 0.677565406518839 Validation accuracy: 0.5128
Epoch: 9 Loss: 0.6603305733837337 Validation accuracy: 0.492
Epoch: 0 Loss: 1.652910146446009 Validation accuracy: 0.4195
Epoch: 1 Loss: 1.4380568154689806 Validation accuracy: 0.4512
Epoch: 2 Loss: 1.4114726588597448 Validation accuracy: 0.4608
Epoch: 3 Loss: 1.314851321204896 Validation accuracy: 0.4651
Epoch: 4 Loss: 1.0151971632798382 Validation accuracy: 0.4743
Epoch: 5 Loss: 1.149806323559166 Validation accuracy: 0.4783
Epoch: 6 Loss: